In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

VEL_LOG_PATH = "/tmp/vel_log.csv"

In [ ]:
df = pd.read_csv(VEL_LOG_PATH)
df["t_rel"] = df["timestamp"] - df["timestamp"].iloc[0]
vel_cols = [c for c in df.columns if c.endswith("_vel")]
df["vel_norm"] = np.linalg.norm(df[vel_cols].values, axis=1)
print(f"Total samples: {len(df)}, Duration: {df['t_rel'].iloc[-1]:.1f}s")
df.head()

## Per-joint velocity over time

In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
axes = axes.flatten()
for i, col in enumerate(vel_cols):
    axes[i].plot(df["t_rel"], df[col], linewidth=0.5)
    axes[i].set_ylabel(col)
    axes[i].axhline(0, color="gray", linewidth=0.3)
axes[-1].plot(df["t_rel"], df["vel_norm"], linewidth=0.5, color="black")
axes[-1].set_ylabel("L2 norm")
for ax in axes[-2:]:
    ax.set_xlabel("time (s)")
fig.suptitle("Joint Velocities Over Time")
plt.tight_layout()
plt.show()

## Timestamp clustering
Cluster by time gaps (idle periods between episodes/actions).

In [ ]:
dt = np.diff(df["timestamp"].values)
GAP_THRESHOLD = np.median(dt) * 5  # gap > 5x median dt = new cluster
gap_indices = np.where(dt > GAP_THRESHOLD)[0] + 1
cluster_ids = np.zeros(len(df), dtype=int)
for idx in gap_indices:
    cluster_ids[idx:] += 1
df["cluster"] = cluster_ids
n_clusters = df["cluster"].nunique()
print(f"Found {n_clusters} clusters (gap threshold: {GAP_THRESHOLD*1000:.1f}ms)")
df.groupby("cluster").agg(
    samples=("t_rel", "count"),
    duration=("t_rel", lambda x: x.max() - x.min()),
    mean_vel_norm=("vel_norm", "mean"),
    max_vel_norm=("vel_norm", "max"),
)

## Velocity distribution per cluster

In [ ]:
fig, axes = plt.subplots(1, min(n_clusters, 6), figsize=(4 * min(n_clusters, 6), 4), sharey=True)
if n_clusters == 1:
    axes = [axes]
for i, (cid, grp) in enumerate(df.groupby("cluster")):
    if i >= 6:
        break
    axes[i].hist(grp["vel_norm"], bins=50, alpha=0.7)
    axes[i].set_title(f"Cluster {cid} (n={len(grp)})")
    axes[i].set_xlabel("vel L2 norm")
axes[0].set_ylabel("count")
fig.suptitle("Velocity Norm Distribution per Cluster")
plt.tight_layout()
plt.show()

## Per-cluster velocity traces

In [ ]:
for cid, grp in df.groupby("cluster"):
    t = grp["t_rel"].values - grp["t_rel"].values[0]
    fig, ax = plt.subplots(figsize=(12, 3))
    for col in vel_cols:
        ax.plot(t, grp[col].values, linewidth=0.5, label=col)
    ax.plot(t, grp["vel_norm"].values, linewidth=1, color="black", label="L2 norm")
    ax.set_title(f"Cluster {cid} — {len(grp)} samples, {t[-1]:.1f}s")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("velocity (rad/s)")
    ax.legend(loc="upper right", fontsize=7)
    plt.tight_layout()
    plt.show()